# 10-зертханалық жұмыс. Күнтізбелік түрлендірулер және іске қосылулардың оқу журналы

**Пән:** Деректерді интеллектуалды талдау  
**Ұзақтығы:** 2 академиялық сағат  
**Алдын ала дайындық:** 10-дәріс материалы; pandas, NumPy, маскалар және CSV сақтау.

## Оқу мақсаттары

- Күндерді екіұштылықсыз тану және пішімдеу.
- Ұзақтықтарды есептеп, күнтізбелік ауысуларды тексеру.
- Синтетикалық UTC журналы бойынша күндік есеп дайындау.

## Ноутбукты іске қосу

Python ортасында мына пакеттерді орнатыңыз: `numpy`, `pandas`. Jupyter бағдарламасында Python 3 ядросын таңдап, ұяшықтарды жоғарыдан төмен орындаңыз. Соңғы код ұяшығында тапсырмаларды өз бетіңізше орындаңыз. Нәтижелерді зертханалық жұмыста көрсетілген `work_results` жолына сақтаңыз.


In [ ]:
from pathlib import Path
import os

# Жұмыс қалтасын анықтау.
project_dir = Path.cwd()
if project_dir.name == "notebooks" and (
    (project_dir.parent / "data" / "football.csv").is_file()
    or (project_dir.parent / "index.html").is_file()
):
    project_dir = project_dir.parent

if Path.cwd() != project_dir:
    os.chdir(project_dir)

print("Жұмыс қалтасы:", project_dir)


## Есептің қойылымы, деректер және қайта орындау

Сіз есептердің орындалуы туралы оқу журналын тексересіз. Мәтіндік күндерді түрлендіріп, ұзақтықтарды есептеу, қателерді анықтау және күндік қорытынды құру қажет. Деректер берілген кодпен толық жасалады. Уақыт белгілері синтетикалық және футболшыларды сипаттамайды. football.csv ішінде бақылау күні жоқ, сондықтан осы жұмыста оған кездейсоқ күндер қосып, нақты тарих ретінде түсіндіруге болмайды.

Бірінші бөлім бастапқы зертханалық жұмыс есептерін сақтайды: strptime, strftime, уақыт белгілері тізімі, секундтар, күндер циклі, timedelta және кібісе жыл. Журнал бөлімі осы дағдыларды кестелік талдаумен байланыстыру үшін қосылған. Тексерілетін жауаптарға ағымдағы уақыт орнына бекітілген күн қолданылады. Қосымша now шақыруын көрсету үшін қолдануға болады, бірақ күтілетін сандардың негізі ретінде емес. Python, pandas және NumPy қажет, интернет керек емес.

Журналдың барлық уақыт белгісі UTC бойынша беріліп, Z әрпімен аяқталады. scheduled_text және finished_text өрістері жоспарлы іске қосылуды және аяқталу мезетін білдіреді. Осы модельде ұзақтық — аяқталу мен жоспар айырмасы. Бұл — жаттығу келісімі: нақты жүйе жоспарды, нақты басталу мен аяқталуды ажыратса, күту уақыты мен орындалу уақыты бөлек есептеледі.

## Журналды құрудың толық коды

Алдымен 2024 жылдың 28 және 29 ақпанын, 1 наурызын қамтитын сағат сайынғы 72 жоспарлы оқиға жасаңыз. Генератор шақыруларының реті бекітілген; қайта орындағанда жаңа rng бастаңыз.



In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import calendar
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
planned = pd.date_range('2024-02-28', periods=72, freq='h', tz='UTC')
seconds = rng.integers(2, 21, size=72)
finished = planned + pd.to_timedelta(seconds, unit='s')
log = pd.DataFrame({
    'event_id': range(72),
    'scheduled_text': planned.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'finished_text': finished.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'source': 'synthetic_training_log',
})



Енді үш ақау және бір дәл қайталану енгізіңіз. Бұл — нақты дереккөзден табылған қасиеттер емес, тапсырманың бөлігі.



In [ ]:
log.loc[5, 'scheduled_text'] = '2024-02-30T05:00:00Z'
log.loc[25, 'finished_text'] = 'bad-date'
negative_end = planned[8] - pd.Timedelta(seconds=3)
log.loc[8, 'finished_text'] = negative_end.strftime(
    '%Y-%m-%dT%H:%M:%SZ'
)
log = pd.concat([log, log.iloc[[10]]], ignore_index=True)
log_original = log.copy(deep=True)
out = Path('work_results') / 'lab10'
out.mkdir(parents=True, exist_ok=True)
assert log.shape == (73, 4)



Журналда event_id=5 үшін жоқ күн, event_id=25 үшін танылмайтын аяқталу және event_id=8 үшін теріс ұзақтық бар. Мазмұны толық бірдей event_id=10 жолы қайталанған. Бұл мәлімет теңгерімді тексеруге мүмкіндік береді; негізгі шешімде мәндерді бастапқы planned және finished арқылы түзетуге болмайды. Шын белгілер қолжетімсіз деп есептеліп, мәселелі жолдар себебі бар жеке карантинге сақталады.

## Тапсырмалардан өзгеше талданған мысал



In [ ]:
example_start = datetime(2024, 12, 31, 23, 59, 50,
                         tzinfo=timezone.utc)
example_end = datetime(2025, 1, 1, 0, 0, 15,
                       tzinfo=timezone.utc)
gap = example_end - example_start
print(gap.total_seconds())
print(example_end.strftime('%d.%m.%Y'))



Нәтиже — 25 секунд және `01.01.2025` жолы. Жыл ауысуын күнтізбелік кітапхана өңдеді. Мезеттер айырмасы timedelta, ал пішімделген нәтиже str болады. Мәтіндік жолдарды азайтуға тырыспаңыз және ұзақтықты тек second өрісінің айырмасымен есептемеңіз: минут ауысқанда бұл тәсіл қате.

## Міндетті тапсырмалар

1. **Тану және шығару.** `2019-07-07T18:59:33` жолын strptime арқылы datetime типіне түрлендіріп, содан кейін `%d.%m.%Y` форматындағы date_format алыңыз. Екі айнымалы типін және уақыт құрамдастарын бөлек көрсетіңіз. strptime неге жолды басқа жолға бір қадамда пішімдемейтінін түсіндіріңіз. Қысқа нәтижені шығаруға дейін бастапқы мезетті сақтаңыз.

2. **Аяқталулар тізімі.** `2019-07-07T18:59:06`, `2019-07-07T19:00:02`, `2019-07-07T19:01:04` белгілерін қолданыңыз. Шартта бұлар дәл үш іске қосылудың аяқталу мезеттері болып табылады. Жоспарлы мезеттер сәйкесінше сол күнгі 18:59:00, 19:00:00 және 19:01:00. datetime_list және report_seconds өрістерін мезеттер айырмасы арқылы құрыңыз. [6, 2, 4] ұзақтықтары және total_time=12 күтіледі. Мұнда тек секундтарды санау кездейсоқ сәйкестік бойынша жарамды, бірақ тәулік шекарасынан өткенде де жұмыс істейтін әдісті көрсетіңіз.

3. **Күндер терезесі және күнтізбе.** 2024-03-02 күнінен есептеу күнін қоспай, алдыңғы үш күнтізбелік күнді өсу ретімен құрыңыз. 2024-02-29 болуы күтіледі. 2023-03-02 үшін қайталап, айырманы түсіндіріңіз. Екі жылдың кібіселігін calendar.isleap арқылы тексеріңіз. Бастапқы мезетке +5 күн және -2 сағат ығысуын бөлек жасаңыз; тексерілетін жауапта оны ағымдағы now мәнімен алмастырмаңыз.

4. **Формат және қате.** Бір күннің екі жазылуын таныңыз: `11.10.2024 12:45:00` және `2024-10-11T12:45:00`. Нысандар теңдігін тексеріңіз. Күн/ай және ай/күн форматтарында `01/02/2024` түсіндірмесі қалай өзгеретінін түсіндіріңіз. Барлық ерекшелікті баспай, try/except ValueError арқылы жоқ 30 ақпанды тексеруді қосыңыз. Жауапта %m пен %M, %H пен %I айырмасын көрсетіңіз.

5. **Қайталануды жою және журналды тану.** log_original көшірмесімен жұмыс істеңіз. event_id=10 қайталануы дәл екенін дәлелдеп, бір нұсқасын сақтаңыз. Егер екі бірдей ID өрістері әртүрлі болса, автоматты жою негізсіз болар еді; осыны түсіндіріңіз. scheduled_text және finished_text өрістерін айқын format және errors='coerce' арқылы UTC бағандарына түрлендіріңіз. Бастапқы мәтінді сақтап, әр бағандағы NaT санын есептеңіз және мәселелі жолдардың event_id мәндерін шығарыңыз.

6. **Ұзақтықтар және карантин.** duration_seconds мәнін .dt.total_seconds() арқылы есептеңіз. Жолдарды дұрыс және карантин жолдарына бөліңіз: қате жоспар, қате аяқталу, теріс ұзақтық. Теріс ұзақтықты тек екі күн де танылған жерде тексеріңіз. reason сақтаңыз. Күтілетін теңгерім: кірістегі 73 жол = 1 дәл қайталану + 3 карантин жолы + 69 дұрыс жол. Құру массивтерінен ештеңе қалпына келтірмеңіз және теріс мәнді модулімен алмастырмаңыз.

7. **UTC және оқу мысалындағы жергілікті күн.** Дұрыс оқиғалардың жоспарлы мезетін timezone(timedelta(hours=5)) арқылы бекітілген +05:00 белдеуінде бейнелеңіз. Жергілікті күнтізбелік күні UTC күнінен ерекшеленетін екі мысал көрсетіңіз. Кері аударғанда мезеттердің өзі сақталғанын дәлелдеңіз. UTC және осы белдеудегі күндік оқиғалар санын салыстырыңыз. Бекітілген оқу ығысуын географиялық уақыт белдеуінің толық тарихы деп ұсынбаңыз.

8. **Күндік есеп және шектер.** Дұрыс оқиғалар бойынша бастапқы үш күнге арналған UTC daily_summary құрыңыз. Аяқталу уақыты бойынша емес, scheduled бойынша топтаңыз. Өрістер: day, n_events, total_seconds, mean_seconds, max_seconds. 28 ақпан, 29 ақпан және 1 наурыз үшін оқиғалар саны 22, 23, 24 болуы тиіс. Олардың қосындысы — 69. 29 ақпан оқиғаларын [басталу, аяқталу) шартымен таңдаңыз. Тиісті күндік санауышпен сәйкестігін тексеріңіз.

9. **Файлдар және дәлелдер.** events_clean.csv, events_quarantine.csv, daily_summary.csv және checks.csv сақтаңыз. checks ішінде тексеруді, күтілетін және нақты нәтижені тізіңіз. Кемінде сегіз тексеру қажет: кіріс пішіні, қайталану, екі NaT санауышы, теріс ұзақтық, жолдар теңгерімі, жалпы күндік көлем және күнтізбелік шекара. Табылған ақаулар, таңдалған ережелер және күндік есептің уақыт белдеуіне тәуелділігі туралы 250-350 сөздік қорытынды жазыңыз.

## Тапсыру форматы және тексеру

lab10.ipynb немесе lab10.py және аталған CSV файлдарын work_results/lab10 ішіне тапсырыңыз. notebook ішінде журналдың бір толық дайындығы болуы, ал барлық тапсырма оның нәтижесін қолдануы тиіс. Басынан қайта орындау сол мәндерді береді. events_clean ішінде event_id, бастапқы мәтіндік өрістер, scheduled, finished, duration_seconds және source қажет. Күндерді айқын белдеуі бар ISO-үйлесімді түрде сақтаңыз; CSV оқылғаннан кейін оларды қайта тану қажет.

events_quarantine ішінде event_id, бастапқы жолдар және reason қажет. Бірнеше белгі қоссаңыз, оларды бөлек сақтаңыз; қорытынды себеп түсінікті болып қалуы тиіс. Карантинді санауға дейін топтауда NaT жолдарын жоғалтпаңыз. Әйтпесе кіріс деректерінің теңгерімі бұзылса да, есеп үйлесімді көрінуі мүмкін.

Сандық агрегаттар үшін күндік есептің total_seconds мәнін дұрыс оқиғалардың duration_seconds қосындысымен салыстырыңыз. Күндер бойынша салмақсыз орташа мән барлық оқиғаның орташа мәніне міндетті түрде тең емес: карантиннен кейін күн көлемдері әртүрлі. «Барлығы» жолын қоссаңыз, оны үш күндік орташа мәннің жай орташасы арқылы емес, бастапқы дұрыс оқиғалардан немесе салмақталған формуламен есептеңіз.

Бастапқы үш белгіні талқылағанда олардың аяқталу мезеттері екені туралы келісімді бөлек атап өтіңіз. Тек нақты басталу мезеттері бойынша орындалу ұзақтығын есептеу мүмкін емес. Бұл оқиғалар іске қосылулар деп аталған бастапқы шарттың елеулі нақтылануы. Тексерілетін талдау уақыт өрісінің мағынасын анықтаудан басталады.

Қосымша тапсырма. Дұрыс scheduled мәндерін Unix секундтарына түрлендіріп, unit='s', utc=True арқылы қалпына келтіріңіз. Мезеттердің теңдігін тексеріңіз. Содан кейін бір сан арқылы миллисекундты қате көрсету қандай мәселе тудыратынын көрсетіңіз. Бастапқы бағандарды алмастырмаңыз және желілік сұраулар орындамаңыз.

## Қорғау сұрақтары

1. Есептеуге дейін пішімдеу неге ақпарат жоғалтуы мүмкін?
2. total_seconds қалдық seconds қасиетінен қалай ерекшеленеді?
3. Белдеуді аудару неге мезетті сақтап, күндік топтауды өзгертеді?
4. Келесі күн шекарасы күндік терезеге неге кірмейді?
5. Аяқталуы жоқ іске қосылу мезеттері бойынша орындалу уақытын табуға бола ма?

## Бағалау және дереккөздер

| Критерий | Балл | Растау |
|---|---|---|
| Форматтар және тізім | 15 | Типтер, [6, 2, 4], қосынды 12 |
| Күнтізбелік операциялар | 15 | Кібіселік, шектер, ығысулар |
| Тану және қайталану | 15 | Айқын format, UTC, дәл қайталану |
| Ұзақтықтар және карантин | 20 | Себептер және 73 = 1 + 3 + 69 теңгерімі |
| Белдеу және күндік есеп | 20 | Мезеттердің сақталуы, 22/23/24 |
| Файлдар және қайта орындау | 10 | CSV, сегіз тексеру |
| Қорғау және қорытынды | 5 | Өріс мағынасы мен шектеулер |
| Барлығы | 100 | |

- [Python: datetime](https://docs.python.org/3/library/datetime.html)
- [Python: calendar](https://docs.python.org/3/library/calendar.html)
- [pandas: уақыт қатарлары](https://pandas.pydata.org/docs/user_guide/timeseries.html)

## Өзіндік жұмыс

Осы зертханалық жұмыстың тапсырмаларын төмендегі ұяшықта орындаңыз. Қажет болса, код пен түсіндірме үшін қосымша ұяшықтар қосыңыз. Есептеулерді аяқтаған соң ядроны қайта іске қосып, ұяшықтарды басынан бастап орындаңыз.

In [ ]:
# Тапсырмаларды орындау кодын осы жерге жазыңыз.
